Random Forest 

Random Forest is ensemblemachine learing algorithm that combine multiple decision tree and usess majority voting (classification ) or averaging (regression) to imporove accuracy and reduce overfitting.

Advantage Handle Large Dataset 
Reduce Overfitting compared yo  a single 


In [ ]:
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score,classification_report


In [ ]:
iris=load_iris()


In [ ]:
X=iris.data


In [ ]:
y=iris.target


In [ ]:
X

In [ ]:
y

In [ ]:
X_train,X_test,y_train,y_test=train_test_split(X,y, test_size=.2, random_state=42)


In [ ]:
X_train,X_test,y_train,y_test = train_test_split(X,y, test_size=.2, random_state=42)

In [ ]:
X_train.shape

In [ ]:
X_test.shape

In [ ]:
rf=RandomForestClassifier(n_estimators=100,max_depth=5,random_state=42)

In [ ]:
rf.fit(X_train,y_train)

In [ ]:
y_pred=rf.predict(X_test)

In [ ]:
accuracy=accuracy_score(y_test,y_pred)


In [ ]:
print(accuracy)

In [ ]:
print(classification_report(y_test,y_pred))

In [ ]:
for feature,importance in zip(iris.feature_names,rf.feature_importances_):
    print(f"{feature}:{importance:.3f}")

In [ ]:
#Zip is reading ek se jyada data sourse ko ek sath read karta hai,

In [ ]:
iris.feature_names

In [ ]:
rf.feature_importances_

In [ ]:
# 1.Create Multiple random Samples from data 
# 2.Train a decision Tree on each Sample
# 3.Each Tree makes a prediction
# 4.Final Prediction is obtained by voting or averaging
# What things it will create 
#Like Loan apacliblity or not , Fraud Detection, Madical Diganosies , Recomendations System , Like Netflix Suggest Movies,
#a)Content Based 
# b)colobrative based
# hybrid is miture of A,B

In [ ]:
import numpy as np
import open3d as o3d
from sklearn.ensemble import RandomForestClassifier

# Example: Train Random Forest
X_train = np.random.rand(100, 3)   # dummy features
y_train = np.random.randint(0, 2, 100)  # dummy labels
rf = RandomForestClassifier()
rf.fit(X_train, y_train)

# Predict on test data
X_test = np.random.rand(50, 3)
y_pred = rf.predict(X_test)

# Convert predictions into 3D points (softcopy model)
points = np.column_stack((X_test, y_pred))

# Create point cloud
pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(points)

# Green wireframe cube
mesh = o3d.geometry.TriangleMesh.create_box(width=1.0, height=1.0, depth=1.0)
mesh.compute_vertex_normals()
mesh.paint_uniform_color([0.0, 1.0, 0.0])  # green

# Visualize
o3d.visualization.draw_geometries([pcd, mesh])


In [ ]:
# Blue wireframe sphere
mesh_blue = o3d.geometry.TriangleMesh.create_sphere(radius=1.0)
mesh_blue.compute_vertex_normals()
mesh_blue.paint_uniform_color([0.0, 0.0, 1.0])  # blue

# Show both together
o3d.visualization.draw_geometries([pcd, mesh_blue])


In [ ]:
!pip install open3d

In [ ]:
print(y_pred)
print(len(y_pred))


In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

fig = plt.figure()
ax = fig.add_subplot(111, projection='3d')

# X_test ke points plot karo
ax.scatter(X_test[:,0], X_test[:,1], X_test[:,2], c=y_pred, cmap='coolwarm')

plt.show()


In [ ]:
import cv2
import numpy as np

# Camera open karo
cam = cv2.VideoCapture(0)

def draw_grid(frame, grid_size=50):
    h, w = frame.shape[:2]
    # Vertical lines
    for x in range(0, w, grid_size):
        cv2.line(frame, (x, 0), (x, h), (0, 255, 0), 1)  # green lines
    # Horizontal lines
    for y in range(0, h, grid_size):
        cv2.line(frame, (0, y), (w, y), (255, 0, 0), 1)  # blue lines
    return frame

while True:
    ret, frame = cam.read()
    if not ret:
        break
    
    # Grid draw karo
    frame = draw_grid(frame, grid_size=80)
    
    cv2.imshow("Camera with Grid", frame)
    
    # ESC key press → exit
    if cv2.waitKey(1) == 27:
        break

cam.release()
cv2.destroyAllWindows()


In [ ]:
import cv2
import torch
import open3d as o3d
import numpy as np
from midas.model_loader import load_model

# 1. Camera capture
cap = cv2.VideoCapture(0)
ret, frame = cap.read()

# 2. Depth estimation (MiDaS model)
model = load_model("DPT_Large")
depth = model.predict(frame)

# 3. Convert depth to point cloud
points = []
h, w = depth.shape
for y in range(h):
    for x in range(w):
        z = depth[y, x]
        points.append([x, y, z])
points = np.array(points)

pcd = o3d.geometry.PointCloud()
pcd.points = o3d.utility.Vector3dVector(points)

# 4. Mesh reconstruction
mesh = o3d.geometry.TriangleMesh.create_from_point_cloud_ball_pivoting(
    pcd, o3d.utility.DoubleVector([0.005, 0.01, 0.02])
)
mesh.compute_vertex_normals()

# 5. Wireframe rendering (Green/Blue)
mesh.paint_uniform_color([0.0, 1.0, 0.0])  # green
o3d.visualization.draw_geometries([mesh])


In [ ]:
!pip install torch torchvision torchaudio

In [ ]:
import cv2
import matplotlib
print("OpenCV version:", cv2.__version__)
print("Matplotlib version:", matplotlib.__version__)


In [ ]:
import cv2

cam = cv2.VideoCapture(0)
ret, frame = cam.read()
cam.release()

# Frame show karo
import matplotlib.pyplot as plt
plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()


In [ ]:
import torch
import cv2
import matplotlib.pyplot as plt

# MiDaS model load karo
midas = torch.hub.load("intel-isl/MiDaS", "DPT_Large")
midas_transforms = torch.hub.load("intel-isl/MiDaS", "transforms")

transform = midas_transforms.default_transform

# Frame ko depth map me convert karo
input_batch = transform(frame).to("cpu")
with torch.no_grad():
    prediction = midas(input_batch)
    depth = torch.nn.functional.interpolate(
        prediction.unsqueeze(1),
        size=frame.shape[:2],
        mode="bicubic",
        align_corners=False,
    ).squeeze().cpu().numpy()

# Depth map show karo
plt.imshow(depth, cmap="plasma")
plt.axis("off")
plt.show()


In [ ]:
import numpy as np
from mpl_toolkits.mplot3d import Axes3D

h, w = depth.shape
X, Y = np.meshgrid(np.arange(w), np.arange(h))
Z = depth

fig = plt.figure(figsize=(8,6))
ax = fig.add_subplot(111, projection='3d')

# Scatter plot with grid lines
ax.plot_wireframe(X[::20,::20], Y[::20,::20], Z[::20,::20], color="green")  # green grid
ax.scatter(X[::50,::50], Y[::50,::50], Z[::50,::50], c="blue", s=2)        # blue points

plt.show()


In [ ]:
import torch
import cv2
import matplotlib.pyplot as plt

# MiDaS model load
midas = torch.hub.load("intel-isl/MiDaS", "DPT_Large")
midas_transforms = torch.hub.load("intel-isl/MiDaS", "transforms")
transform = midas_transforms.default_transform

# Camera frame capture
cam = cv2.VideoCapture(0)
ret, frame = cam.read()
cam.release()

# Depth prediction
input_batch = transform(frame).to("cpu")
with torch.no_grad():
    prediction = midas(input_batch)
    depth = torch.nn.functional.interpolate(
        prediction.unsqueeze(1),
        size=frame.shape[:2],
        mode="bicubic",
        align_corners=False,
    ).squeeze().cpu().numpy()

# Show depth map
plt.imshow(depth, cmap="plasma")
plt.axis("off")
plt.show()


In [ ]:
!pip install timm

In [ ]:
import numpy as np
from mpl_toolkits.mplot3d import Axes3D
import matplotlib.pyplot as plt

h, w = depth.shape
X, Y = np.meshgrid(np.arange(w), np.arange(h))
Z = depth

fig = plt.figure(figsize=(8,6))
ax = fig.add_subplot(111, projection='3d')

# Wireframe grid (green)
ax.plot_wireframe(X[::20,::20], Y[::20,::20], Z[::20,::20], color="green")

# Scatter points (blue)
ax.scatter(X[::50,::50], Y[::50,::50], Z[::50,::50], c="blue", s=2)

plt.show()


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# 1. Load image (mobile camera photo)
img = cv2.imread("glass.jpg")
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

# 2. Edge detection
edges = cv2.Canny(gray, 100, 200)

# 3. Show wireframe (2D outline)
plt.imshow(edges, cmap="gray")
plt.title("Wireframe Outline")
plt.axis("off")
plt.show()
